# Sequential PF Submission

Particle Filter with sequence awareness, RTS smoothing, and multi-feature likelihood.

In [ ]:
import numpy as np
import pandas as pd
from glob import glob
import os
from tqdm import tqdm

In [ ]:
# Config
DATA_DIR = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'
OUTPUT_PATH = '/kaggle/working/submission.csv'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, 'sample_submission.csv')

# PF hyperparameters
N_PARTICLES = 200
N_SEEDS = 64
MOM = 0.998
VN = 0.002
RN = 0.002
PN = 0.005
RP = 0.1
RR = 0.001
RESAMP_THRESH = 0.5
TRIM_RATIO = 0.3
LIK_TEMPERATURE = 3.0
FIXED_LAG = 10
INCL_ENABLED = False

In [ ]:
def compute_inclination(hw):
    """Compute well inclination from XYZ coordinates."""
    n = len(hw)
    incl = np.full(n, 90.0)
    if 'X' not in hw.columns or 'Z' not in hw.columns:
        return incl
    x = hw['X'].values
    z = hw['Z'].values
    valid = np.isfinite(x) & np.isfinite(z)
    if valid.sum() < 2:
        return incl
    dx = np.diff(x)
    dz = np.diff(z)
    angles = np.abs(np.arctan2(dz, dx)) * 180.0 / np.pi
    for i in range(len(angles)):
        if np.isfinite(angles[i]):
            incl[i + 1] = angles[i]
    return incl

In [ ]:
def compute_azimuth(hw):
    """Compute well azimuth from XYZ coordinates."""
    n = len(hw)
    az = np.full(n, 0.0)
    if 'X' not in hw.columns or 'Y' not in hw.columns:
        return az
    x = hw['X'].values
    y = hw['Y'].values
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 2:
        return az
    dx = np.diff(x)
    dy = np.diff(y)
    angles = np.arctan2(dy, dx) * 180.0 / np.pi
    angles = (angles + 360) % 360
    for i in range(len(angles)):
        if np.isfinite(angles[i]):
            az[i + 1] = angles[i]
    return az

In [ ]:
def run_particle_filter(hw, tw, n_particles=N_PARTICLES, seed=42,
                        use_multi_feature=False, record_history=False):
    """Conservative Particle Filter for TVT tracking via GR matching."""
    tw_s = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0

    last = kn.iloc[-1]
    last_tvt = float(last['TVT_input'])
    last_Z = float(last['Z'])
    last_MD = float(last['MD'])

    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))

    feat_sigmas = {'GR': gs}
    if use_multi_feature:
        for feat in ['X', 'Y', 'Z']:
            if feat in tw_s.columns:
                vals = tw_s[feat].fillna(tw_s[feat].mean()).values.astype(float)
                feat_sigmas[feat] = np.std(vals) * 0.5 + 1e-6
        if 'incl' in tw_s.columns:
            feat_sigmas['incl'] = 15.0
        if 'azimuth' in tw_s.columns:
            feat_sigmas['azimuth'] = 30.0

    if len(kn) >= 3:
        tail = kn.tail(30)
        dt = np.diff(tail['TVT_input'].values)
        dz = np.diff(tail['Z'].values)
        dm = np.diff(tail['MD'].values)
        m = dm > 0
        ir = float(np.median((dt[m] + dz[m]) / dm[m])) if m.sum() >= 3 else 0.0
    else:
        ir = 0.0

    N = n_particles
    rng = np.random.default_rng(seed)
    pos = last_tvt + last_Z + 2.0 * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N) / N

    md_v = ev['MD'].values.astype(float)
    z_v = ev['Z'].values.astype(float)
    gr_v = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values[ev.index].astype(float)

    obs_features = {'GR': gr_v}
    if use_multi_feature:
        for feat in ['X', 'Y', 'Z']:
            if feat in hw.columns:
                vals = hw[feat].interpolate(limit_direction='both').fillna(
                    hw[feat].mean()).values
                obs_features[feat] = vals[ev.index].astype(float)
        incl = compute_inclination(hw)
        az = compute_azimuth(hw)
        obs_features['incl'] = incl[ev.index].astype(float)
        obs_features['azimuth'] = az[ev.index].astype(float)

    particle_history = [] if record_history else None
    weight_history = [] if record_history else None

    res = np.empty(len(ev))
    prev_MD = last_MD
    prev_tvt = last_tvt
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + RN * rng.standard_normal(N)
        pos = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - z_v[i], tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos = tvt_p + z_v[i]
        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        lk = np.maximum(np.exp(-0.5 * np.minimum(((gr_v[i] - eg) / gs) ** 2, 600.)), 1e-300)

        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w = w * lk
        ws = w.sum()
        w = w / ws if ws > 0 else np.ones(N) / N

        if record_history:
            particle_history.append(pos.copy())
            weight_history.append(w.copy())

        n_eff = 1.0 / (w**2).sum()
        if n_eff < RESAMP_THRESH * N:
            cum = np.cumsum(w)
            idx = np.clip(np.searchsorted(cum, rng.uniform(0, 1.0 / N) + np.arange(N) / N), 0, N - 1)
            pos = pos[idx] + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w = np.ones(N) / N
            if record_history:
                particle_history[-1] = pos.copy()
                weight_history[-1] = w.copy()

        tvt_i = float(np.dot(w, pos - z_v[i]))
        if abs(tvt_i - prev_tvt) > 2.0:
            tvt_i = prev_tvt + np.sign(tvt_i - prev_tvt) * 2.0
        res[i], prev_tvt, prev_MD = tvt_i, tvt_i, md_v[i]

    out = hw['TVT_input'].values.astype(float).copy()
    out[list(ev.index)] = res

    if record_history:
        return out, log_lik, particle_history, weight_history

    return out, log_lik

In [ ]:
def apply_rts_smoother(particle_hist, weight_hist):
    """RTS backward smoother."""
    if not particle_hist or len(particle_hist) < 2:
        return particle_hist
    T = len(particle_hist)
    smoothed = [None] * T
    smoothed[-1] = particle_hist[-1].copy()
    for t in range(T - 2, -1, -1):
        p_curr = particle_hist[t]
        p_next = particle_hist[t + 1]
        p_smooth = smoothed[t + 1]
        mean_curr = p_curr.mean()
        mean_next = p_next.mean()
        cov = np.cov(p_curr, p_next) if len(p_curr) > 1 else np.eye(2) * 0.1
        var_next = np.var(p_next) + 1e-6
        G = cov[0, 1] / var_next
        smoothed[t] = p_curr + G * (p_smooth - mean_next)
    return smoothed

In [ ]:
def apply_fixed_lag_smoother(particle_hist, weight_hist, lag=FIXED_LAG):
    """Fixed-lag smoother."""
    if not particle_hist or len(particle_hist) < lag + 1:
        return particle_hist
    T = len(particle_hist)
    smoothed = [p.copy() for p in particle_hist]
    for t in range(lag, T):
        window = particle_hist[t - lag:t + 1]
        weights = weight_hist[t - lag:t + 1]
        avg = np.mean(window, axis=0)
        smoothed[t] = 0.7 * smoothed[t] + 0.3 * avg
    return smoothed

In [ ]:
def predict_pf_sequential(hw, tw, use_rts=True, use_fixed_lag=True,
                          use_multi_feature=True):
    """Sequence-aware PF with smoothing."""
    known_mask = hw['TVT_input'].notna().values
    predict_mask = ~known_mask
    if predict_mask.sum() < 1:
        return hw['TVT_input'].values.astype(float).copy(), -1e9

    ev = hw[hw['TVT_input'].isna()]
    tw_tvt = tw['TVT'].values.astype(float)

    all_results = []
    all_liks = []

    for s in range(N_SEEDS):
        try:
            result, log_lik, particle_hist, weight_hist = run_particle_filter(
                hw, tw, seed=s,
                use_multi_feature=use_multi_feature,
                record_history=True
            )
            if particle_hist and weight_hist and len(particle_hist) > 0:
                if use_rts:
                    particle_hist = apply_rts_smoother(particle_hist, weight_hist)
                if use_fixed_lag:
                    particle_hist = apply_fixed_lag_smoother(particle_hist, weight_hist)
                z_vals = ev['Z'].values
                smoothed = np.array([np.dot(weight_hist[t], particle_hist[t] - z_vals[t]) 
                                    for t in range(len(particle_hist))])
                result = hw['TVT_input'].values.astype(float).copy()
                result[list(ev.index)] = smoothed

            all_results.append(result)
            all_liks.append(log_lik)
        except Exception as e:
            print(f'    Error seed {s}: {e}')
            all_results.append(hw['TVT_input'].fillna(
                hw['TVT_input'].mean() if known_mask.any() else 0.0
            ).values.astype(float))
            all_liks.append(-1e9)

    arr = np.stack(all_results, 0)
    liks = np.array(all_liks)
    n = len(liks)
    n_trim = int(np.ceil(n * TRIM_RATIO))
    valid_mask = liks > -1e8

    if valid_mask.sum() >= 2:
        order = np.argsort(liks[valid_mask])
        keep_count = max(2, len(order) - 2 * n_trim)
        keep_idx = np.where(valid_mask)[0][order[:keep_count]]
    else:
        keep_idx = np.where(valid_mask)[0]

    if len(keep_idx) == 0:
        return hw['TVT_input'].values.astype(float).copy(), -1e9

    arr_trimmed = arr[keep_idx]
    liks_trimmed = liks[keep_idx]
    w = np.exp((liks_trimmed - liks_trimmed.max()) / LIK_TEMPERATURE)
    w /= w.sum()
    result = (w[:, None] * arr_trimmed).sum(axis=0)
    result[known_mask] = hw.loc[known_mask, 'TVT_input'].values.astype(float)
    result = np.clip(result, tw_tvt.min() - 200, tw_tvt.max() + 200)
    result = np.nan_to_num(result, nan=0.0, posinf=tw_tvt.max() + 200, neginf=tw_tvt.min() - 200)

    return result, float(liks_trimmed.max())

In [ ]:
# Generate submission
print('Generating submission...')
sub = pd.read_csv(SAMPLE_SUB_PATH)
predictions = {}
test_files = sorted(glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))

for f in tqdm(test_files, desc='[Predict] Wells'):
    wid = os.path.basename(f).replace('__horizontal_well.csv', '')
    hw = pd.read_csv(f)
    tw = pd.read_csv(os.path.join(TEST_DIR, f'{wid}__typewell.csv'))
    print(f'\nPredicting {wid}: {len(hw)} rows, {hw["TVT_input"].notna().sum()} known')
    tvt_pred, _ = predict_pf_sequential(
        hw, tw,
        use_rts=True,
        use_fixed_lag=True,
        use_multi_feature=True
    )
    predictions[wid] = np.asarray(tvt_pred, dtype=float)
    print(f'  TVT: [{tvt_pred.min():.1f}, {tvt_pred.max():.1f}], mean={tvt_pred.mean():.1f}')

for idx, row in sub.iterrows():
    wid = row['id'].rsplit('_', 1)[0]
    ri = int(row['id'].rsplit('_', 1)[1])
    if wid in predictions:
        sub.at[idx, 'tvt'] = predictions[wid][ri]

sub.to_csv(OUTPUT_PATH, index=False)
print(f'\nSubmission saved: {OUTPUT_PATH} ({len(sub)} rows)')
print(f'NaN count: {sub["tvt"].isna().sum()}')